<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">4. Working with Complex Data</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 4.2 Demo - Working with Complex Data Types in Spark
### Duration - 15 Minutes
This demonstration works with the `raw_customer_profiles` table created during the required setup - Bakehouse customer data where the profile, products, and purchase history arrive as JSON strings, the way data typically lands from an API or event feed. We convert the JSON to structured types and analyze it with array, struct, and collection functions.

## Learning Objectives

**By the end of this demo, you will be able to:**

- Infer a schema from sample JSON with `schema_of_json()` and convert JSON strings to structured types with `from_json()`
- Access struct fields using dot notation and `getField()`
- Unnest arrays with `explode()` and filter them with `array_contains()`
- Aggregate values into collections with `collect_list()` and `collect_set()`
- Reshape data with the `pivot()` method

## REQUIRED - SELECT CLASSIC COMPUTE

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<div>
<strong style="color: #c62828">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333">Before starting this notebook, select your <strong>Classic Compute</strong> cluster from the compute drop-down at the top-right of the notebook. Be aware that <strong>Serverless</strong> is selected by default.</p>
<ul style="margin: 12px 0 0 16px; color: #333">
<li>If your cluster is not shown in the drop-down, select <strong>More</strong>, then choose your cluster in the <strong>Attach to an existing compute resource</strong> dialog.</li>
<li>If your cluster has terminated, open <strong>Compute</strong> in the left navigation, start your cluster, then return here and attach it.</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333"><strong>NOTE:</strong> This course was developed and tested on <strong>Databricks Runtime 17.3 LTS (Apache Spark 4.0)</strong>. Several demonstrations reference the Spark UI and query execution plans, which are not exposed on Serverless compute.</p>
</div>
</div>
</div>

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It validates that the **01_Demo_Required_Setup** notebook has been run (including the raw complex data tables), sets your default catalog, and sets the default schema to `spark_dev`.

In [0]:
%run ../Includes/Classroom-Setup-4

## B. Load and Inspect the Raw Data

First, let's look at what we received.

In [0]:
%python
# Read the Raw Customer Profiles Table
raw_profiles_df = spark.read.table("raw_customer_profiles")

In [0]:
%python
# Print the Schema
raw_profiles_df.printSchema()

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Every column is a plain <code>string</code> - including <code>profile</code> (a JSON object), <code>products</code> (a JSON array of strings), and <code>purchases</code> (a JSON array of objects). Until we convert them, Spark cannot reason about anything inside those strings.</p>
</div>

In [0]:
%python
# Visually Inspect a Subset of the Data
display(raw_profiles_df.limit(5))

## C. Convert JSON Strings to Structured Types

Why convert?

- **Parsing overhead** - every query against a JSON string re-parses it
- **Type safety** - structs have typed, named fields the engine validates
- **Query optimization** - the optimizer can prune and push down struct fields, not strings

The conversion recipe: sample a value, infer its schema with `schema_of_json()`, then apply `from_json()`.

In [0]:
%python
# Sample One Value from Each JSON Column
profile_sample = raw_profiles_df.select("profile").limit(1).collect()[0][0]
products_sample = raw_profiles_df.select("products").limit(1).collect()[0][0]
purchases_sample = raw_profiles_df.select("purchases").limit(1).collect()[0][0]

print(f"profile:   {profile_sample}")
print(f"products:  {products_sample}")
print(f"purchases: {purchases_sample[:120]}...")

In [0]:
%python
# Infer the Schemas from the Samples
from pyspark.sql.functions import schema_of_json, lit

# schema_of_json() infers a DDL schema string from a literal JSON example
display(
    raw_profiles_df.select(
        schema_of_json(lit(profile_sample)).alias("profile_schema"),
        schema_of_json(lit(products_sample)).alias("products_schema"),
        schema_of_json(lit(purchases_sample)).alias("purchases_schema")
    ).limit(1)
)

In [0]:
%python
# Convert the JSON Strings with from_json
from pyspark.sql.functions import col, from_json, schema_of_json, lit

profiles_df = raw_profiles_df.select(
    "customerID",
    "name",
    "email",
    from_json(col("profile"), schema_of_json(lit(profile_sample))).alias("profile"),
    from_json(col("products"), schema_of_json(lit(products_sample))).alias("products"),
    from_json(col("purchases"), schema_of_json(lit(purchases_sample))).alias("purchases")
)

profiles_df.printSchema()

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">The schema now shows real structure: <code>profile</code> is a <code>struct</code> with named fields, <code>products</code> is an <code>array&lt;string&gt;</code>, and <code>purchases</code> is an <code>array</code> of <code>struct</code> elements. Everything downstream gets simpler and faster.</p>
</div>

## D. Referencing Struct Fields

Struct fields are accessed with dot notation or `getField()` - both produce the same plan.

In [0]:
%python
# Access Struct Fields
from pyspark.sql.functions import col

display(
    profiles_df.select(
        "name",
        col("profile.city"),                            # Dot notation
        col("profile").getField("country").alias("country"),  # getField()
        col("profile.continent")
    ).limit(10)
)

## E. Working with Arrays

Array functions answer questions without unnesting: how many distinct products has each customer bought, and who buys a specific product?

In [0]:
%python
# Measure and Filter Arrays
from pyspark.sql.functions import col, array_size, array_contains, desc

display(
    profiles_df
    .select(
        "name",
        array_size(col("products")).alias("distinct_products"),
        array_size(col("purchases")).alias("purchase_count"),
        "products"
    )
    .orderBy(desc("purchase_count"))
    .limit(10)
)

In [0]:
%python
# Find Customers Who Bought a Specific Product
from pyspark.sql.functions import col, array_contains

golden_gate_fans_df = profiles_df.filter(
    array_contains(col("products"), "Golden Gate Ginger")
)

print(f"{golden_gate_fans_df.count()} customers bought Golden Gate Ginger")
display(golden_gate_fans_df.select("name", "products").limit(10))

### Unnesting with explode()

To analyze the individual purchases we unnest the `purchases` array: one output row per array element, with the other columns preserved.

In [0]:
%python
# Explode the Purchases Array
from pyspark.sql.functions import col, explode

purchases_exploded_df = profiles_df.select(
    "customerID",
    "name",
    col("profile.country").alias("country"),
    explode(col("purchases")).alias("purchase")
)

display(purchases_exploded_df.limit(10))

In [0]:
%python
# Flatten the Purchase Struct
from pyspark.sql.functions import col

purchase_details_df = purchases_exploded_df.select(
    "customerID",
    "name",
    "country",
    col("purchase.product").alias("product"),
    col("purchase.quantity").alias("quantity"),
    col("purchase.totalPrice").alias("totalPrice"),
    col("purchase.paymentMethod").alias("paymentMethod")
)

display(purchase_details_df.limit(10))

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Each customer row became one row per purchase - the row count multiplied by the average array length. On large arrays this explosion is the main performance hazard of <code>explode()</code>.</p>
</div>

## F. Aggregating to Collections

`collect_list()` and `collect_set()` are the inverse of `explode()`: they gather rows back into arrays. Let's build each country's product catalog.

In [0]:
%python
# Collect Products per Country
from pyspark.sql.functions import collect_list, collect_set, array_size, col, desc

country_products_df = (
    purchase_details_df
    .groupBy("country")
    .agg(
        collect_list("product").alias("all_purchases"),   # keeps duplicates
        collect_set("product").alias("unique_products")   # unique only
    )
    .withColumn("purchase_count", array_size(col("all_purchases")))
    .withColumn("product_count", array_size(col("unique_products")))
)

display(country_products_df.orderBy(desc("purchase_count")).limit(10))

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333"><code>all_purchases</code> holds one element per purchase (duplicates included), while <code>unique_products</code> holds each product once. When duplicates and order don't matter, prefer <code>collect_set()</code> - it collects and shuffles less data.</p>
</div>

## G. Using the pivot Method

`pivot()` turns row values into columns - useful for cross-tabulation and feature engineering. Let's count purchases by country and payment method.

In [0]:
%python
# Pivot Payment Methods by Country
payment_pivot_df = (
    purchase_details_df
    .groupBy("country")
    .pivot("paymentMethod")
    .count()
    .fillna(0)
)

display(payment_pivot_df)

<div style="font-size: 1em; border-left: 4px solid #1B5162; background: #F9F7F4; padding: 12px 16px; border-radius: 4px; margin: 12px 0">
<strong style="color: #1B3139">What to Observe</strong>
<p style="margin: 8px 0 0 0; color: #333">Each distinct <code>paymentMethod</code> value became a column. <code>fillna(0)</code> replaces the nulls produced where a country had no purchases with that method. Pivoting on a high-cardinality column produces very wide results - constrain the values with <code>pivot(col, [values])</code> when you know them.</p>
</div>

## Key Takeaways

**Struct operations:**

- Convert JSON strings to structs early: sample, `schema_of_json()`, `from_json()`
- Access fields with dot notation or `getField()`

**Array operations:**

- `array_size()`, `array_contains()`, and `element_at()` answer questions without unnesting
- `explode()` unnests arrays into rows; watch the row multiplication on large arrays

**Complex aggregations:**

- `collect_list()` / `collect_set()` gather rows into arrays - the inverse of `explode()`
- `pivot()` turns row values into columns for cross-tabulation

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>